<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study_Practice/08_Dog_breed_identification_Practice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

# 3. 캐글 API로 MNIST 데이터 초고속 다운로드 및 압축 해제
!pip install -q kaggle

Mounted at /content/drive


In [ ]:
!kaggle competitions download -c dog-breed-identification

100% 691M/691M [00:04<00:00, 155MB/s]



In [ ]:
!unzip -q dog-breed-identification.zip -d ./dog_breed

!ls -l ./dog_breed

total 26356
-rw-r--r-- 1 root root   482063 Dec 11  2019 labels.csv
-rw-r--r-- 1 root root 25200295 Dec 11  2019 sample_submission.csv
drwxr-xr-x 2 root root   659456 May 30 11:27 test
drwxr-xr-x 2 root root   634880 May 30 11:27 train


In [ ]:
import warnings
import os

# 멀티프로세싱 관련 불필요한 OS 에러/경고 무시
warnings.filterwarnings('ignore', category=UserWarning)
# 파이썬의 복잡한 멀티프로세싱 에러 로그 출력을 방지하기 위해 환경 변수 설정 (필요시)
os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
import numpy as np
import pandas as pd

train_df = pd.read_csv('./dog_breed/labels.csv')
print(len(train_df))
# print(train_df.head()) # id, breed

unique_breeds = train_df['breed'].unique()
unique_breeds.sort()

breed_to_idx = {breed: idx for idx, breed in enumerate(unique_breeds)}

train_df['label_idx'] = train_df['breed'].map(breed_to_idx)
train_df['img_file'] = train_df['id'] + '.jpg'

print(train_df[['img_file', 'label_idx', 'breed']].head())

10222
                               img_file  label_idx             breed
0  000bec180eb18c7604dcecc8fe0dba07.jpg         19       boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97.jpg         37             dingo
2  001cdf01b096e06d78e9e5112d419397.jpg         85          pekinese
3  00214f311d5d2247d5dfe4fe24b2303d.jpg         15          bluetick
4  0021f9ceb3235effd7fcde7f7538ed62.jpg         49  golden_retriever


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class DogBreedDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['img_file']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label = self.df.iloc[idx]['label_idx']

        if self.transform:
            image = self.transform(image)

        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_img_dir = './dog_breed/train'
full_dataset = DogBreedDataset(df=train_df, img_dir=train_img_dir, transform=train_transform)
print(len(full_dataset))

10222


In [ ]:
from torch.utils.data import DataLoader, random_split

# train_size = int(0.8 * len(full_dataset))
# val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [0.8, 0.2])
# print(len(train_dataset), len(val_dataset)) # 8178 2044

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([64, 3, 224, 224]) torch.Size([64])


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 120)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 74.0MB/s]


cuda


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import time

epochs = 3
for epoch in range(epochs):
    start_time = time.time()

    model.train()
    train_loss, train_correct, train_total = 0.0, 0.0, 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_correct += (predicted == labels).sum().item()
        train_total += images.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total * 100

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += (predicted == labels).sum().item()
            val_total += images.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total * 100

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    print(f"Epoch [{epoch+1}/{epochs}] | Time: {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}\n")


Epoch [1/3] | Time: 1m 44s
Train Loss: 0.3045 | Train Acc: 92.49
Val Loss: 0.5368 | Val Acc: 83.66

Epoch [2/3] | Time: 1m 43s
Train Loss: 0.1640 | Train Acc: 96.22
Val Loss: 0.5538 | Val Acc: 83.46

Epoch [3/3] | Time: 1m 42s
Train Loss: 0.1098 | Train Acc: 97.60
Val Loss: 0.5629 | Val Acc: 83.76



In [ ]:
import glob

class KaggleTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.image_paths = glob.glob(os.path.join(test_dir, '*.jpg')) + \
                           glob.glob(os.path.join(test_dir, '*.jpeg')) + \
                           glob.glob(os.path.join(test_dir, '*.png'))
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        image_id = os.path.basename(img_path).split('.')[0]

        if self.transform:
            image = self.transform(image)

        return image, image_id

```
image_id = os.path.basename(img_path).split('.')[0]

os.path.basename(img_path): 전체 파일 경로에서 폴더 경로 부분을 제거하고,
가장 마지막에 있는 순수 파일명(+확장자)만 리턴해주는 함수입니다.

만약 파일명 자체에 점(.)이 여러 개 들어간 경우(예: dog.version.1.jpg)라면
.split('.')[0]은 dog만 가져오는 버그가 생깁니다.
이럴 때는 os.path.splitext()를 쓰면 마지막 확장자만 도려낼 수 있습니다.

filename, ext = os.path.splitext(os.path.basename('/content/test/dog.version.1.jpg'))

glob.glob은 해당 패턴에 매칭되는 실제 파일 경로들을 리스트 형태로 가져옵니다.
파이썬에서 더하기(+) 연산을 하면 자바의 List.addAll()처럼 두 리스트가 하나로 합쳐집니다.
백슬래시(\)는 코드가 너무 길어져서 아랫줄로 개행할 때 쓰는 줄바꿈 기호입니다.
```

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

TEST_DIR = './dog_breed/test'

test_dataset = KaggleTestDataset(test_dir=TEST_DIR, transform=test_transform)
print(len(test_dataset))

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

10357


In [ ]:
images, image_ids = next(iter(test_loader))
print(images.shape)
print(image_ids[:5])

torch.Size([64, 3, 224, 224])
('21ca3205f590adf0f266ef71323bba49', 'fc9f8662172cd0fd27a6709f93034138', '78a72d0e1953c906d3803da6c1798466', '16e979f5cd47a91d0c9aa629668b3487', 'c559e7b7341b152df3c3559d61abe9c9')


In [ ]:
# @title
import torch.nn.functional as F

model.eval()

all_probs = []
all_image_ids = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_image_ids.extend(image_ids)

all_probs = np.vstack(all_probs)

sample_sub = pd.read_csv('./dog_breed/sample_submission.csv')
breed_columns = sample_sub.columns[:1]

submission_df = pd.DataFrame(all_probs, columns=breed_columns)
submission_df.insert(0, 'id', all_image_ids)

print(submission_df.shape)
print(submission_df.head(3))

submission_df.to_csv('submission(Sample reference ver).csv', index=False)

In [ ]:
import torch.nn.functional as F

all_probs = []
all_image_ids = []

model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())# 처음부터 extend하면 어떻게 될까? 뒤죽박죽?
        all_image_ids.extend(image_ids)

final_probs_array = np.vstack(all_probs)

submission_df = pd.DataFrame(final_probs_array, columns=unique_breeds)

submission_df.insert(0, 'id', all_image_ids)

submission_df.to_csv('submission.csv', index=False)